## Advanced Secrets Management

# Advanced Google Cloud Secret Manager: Inventory, Metadata Taxonomies, and Purge Operations

In previous lessons, we explored the fundamentals of managing sensitive information using Google Cloud's secrets management services. In this lesson, we will dive into advanced features that help you organize, control, and manage the lifecycle of your secrets more effectively.

We will cover how to inventory all managed assets via project listings, use key-value metadata labels for multi-tenant categorization, audit historical version lineages, and differentiate between soft container exclusions and permanent cryptographic destruction. These capabilities are essential for maintaining rigorous compliance, tight security boundaries, and operational efficiency in your cloud environment.

---

## Listing All Project Secrets

Google Cloud Secret Manager allows you to query and compile an inventory of all secrets within a designated project. This operational visibility is crucial for automated compliance audits, rotative state scripting, and orphan resource cleanup.

```python
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Replace with your Google Cloud project ID
project_id = "your-project-id"
parent = f"projects/{project_id}"

print("Inventory Check: Scanning project secrets...")
# List all secrets in the project
for secret in client.list_secrets(request={"parent": parent}):
    print("🔹 Secret Name:", secret.name)

```

**Console Output:**

```text
🔹 Secret Name: projects/your-project-id/secrets/api-key
🔹 Secret Name: projects/your-project-id/secrets/database-password
🔹 Secret Name: projects/your-project-id/secrets/my-secret

```

This request returns a paginated iterator stream containing the metadata structures of all secrets matching the designated parent project ID coordinates.

---

## Labeling and Unlabeling Secrets (Metadata Taxonomies)

Labels in Google Cloud Secret Manager are key-value string pairs that help you categorize, filter, and allocate resource costs for your secrets. You can modify labels on the fly to tie secrets to specific runtime environments (`env: production`), organizational units (`team: devops`), or microservice dimensions (`app: authentication`).

### 1. Appending and Modifying Labels

```python
from google.cloud import secretmanager
from google.protobuf import field_mask_pb2

client = secretmanager.SecretManagerServiceClient()
secret_name = f"projects/{project_id}/secrets/my-secret"

# Define the new structural metadata labels
labels = {
    "environment": "production",
    "team": "devops"
}

# Construct the payload dict matching the Secret proto spec
secret_update_payload = {
    "name": secret_name, 
    "labels": labels
}

# Target ONLY the labels dictionary to protect other configuration fields
update_mask = field_mask_pb2.FieldMask(paths=["labels"])

updated_secret = client.update_secret(
    secret=secret_update_payload, 
    update_mask=update_mask
)
print("Updated Labels:", updated_secret.labels)

```

**Console Output:**

```text
Updated Labels: {'environment': 'production', 'team': 'devops'}

```

> ⚙️ **Architectural Deep-Dive: The `update_mask` (FieldMask)**
> The `update_mask` parameter is a critical architectural guardrail in this transaction. It functions as a surgical filter specifying precisely which fields inside the backend storage object should be overwritten. By setting `paths=["labels"]`, you explicitly instruct the Google Cloud API gateway to modify **only** the taxonomy block. Other structural properties assigned to that secret container (such as multi-region data replication profiles, automatic expiration TTLs, or customer-managed encryption key rings) remain entirely un-touched.

### 2. Dropping (Removing) a Key Label

To delete a specific tag entry entirely, simply omit the target key from your input dictionary map and submit the modification using the same field mask process:

```python
# Omit the 'team' key to strip it out of the metadata layer
cleared_labels = {
    "environment": "production"
}

secret_update_payload = {
    "name": secret_name, 
    "labels": cleared_labels
}

update_mask = field_mask_pb2.FieldMask(paths=["labels"])

updated_secret = client.update_secret(
    secret=secret_update_payload, 
    update_mask=update_mask
)
print("Labels after removal:", updated_secret.labels)

```

**Console Output:**

```text
Labels after removal: {'environment': 'production'}

```

---

## Working with Secret Versions

Each secret container maintains an immutable historical ledger of version records. This makes secret rotations highly safe, as past connection strings or credentials remain preserved if an application rollback is triggered.

### 1. Auditing Version States

You can scan the structural status ledger of all versions nested inside a target container:

```python
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient()
secret_name = f"projects/{project_id}/secrets/my-secret"

print(f"Auditing history for: {secret_name}")
# List all versions of the target secret
for version in client.list_secret_versions(request={"parent": secret_name}):
    print(f" ├─ Version ID Path: {version.name} | State: {version.state.name}")

```

**Console Output:**

```text
 ├─ Version ID Path: projects/your-project-id/secrets/my-secret/versions/3 | State: ENABLED
 ├─ Version ID Path: projects/your-project-id/secrets/my-secret/versions/2 | State: ENABLED
 ├─ Version ID Path: projects/your-project-id/secrets/my-secret/versions/1 | State: ENABLED

```

### 2. Extracting Target Payloads (Latest vs. Explicit Versions)

Consumer applications can request the runtime data string using either the dynamic `latest` alias index pointer or a specific historic iteration integer key:

```python
# Option A: Pull data using the global convenience alias pointer
latest_version_name = f"{secret_name}/versions/latest"
response_latest = client.access_secret_version(request={"name": latest_version_name})
print("Latest Secret Value:", response_latest.payload.data.decode("UTF-8"))

# Option B: Pull the historical genesis state block explicitly targeting Version 1
version_1_name = f"{secret_name}/versions/1"
response_v1 = client.access_secret_version(request={"name": version_1_name})
print("Version 1 Secret Value:", response_v1.payload.data.decode("UTF-8"))

```

**Console Output:**

```text
Latest Secret Value: my-updated-secret-value
Version 1 Secret Value: my-original-secret-value

```

---

## Secret Deletion vs. Destruction

When decommissioning infrastructure, it is critical to understand the distinction between deleting an entire secret container resource and destroying a standalone, isolated payload iteration. Both actions are completely **permanent and irreversible**.

### 1. Deleting an Entire Secret Container

Deleting a secret executes a top-level **cascading wipe**. This deletes the primary tracking container shell, all metadata tags, IAM bindings, and **every single historical version entry** stored within that path.

```python
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient()
secret_name = f"projects/{project_id}/secrets/my-secret"

# Permanent cascading deletion operation
client.delete_secret(request={"name": secret_name})
print(f"🔥 Core container '{secret_name}' and all inner versions permanently deleted.")

```

**Console Output:**

```text
🔥 Core container 'projects/your-project-id/secrets/my-secret' and all inner versions permanently deleted.

```

### 2. Destroying an Isolated Secret Version

If you want to sanitize old, stale, or compromised credentials without breaking your application's current environment connection strings, you can **destroy** a specific version layer. This permanently purges the data payload bits for that explicit index while keeping the parent wrapper and other active version indexes intact.

```python
target_version_path = f"projects/{project_id}/secrets/my-secret/versions/1"

# Permanently shred the secret data payload for version 1
client.destroy_secret_version(request={"name": target_version_path})
print(f"💥 Cryptographic payload for version '{target_version_path}' destroyed.")

```

**Console Output:**

```text
💥 Cryptographic payload for version 'projects/your-project-id/secrets/my-secret/versions/1' destroyed.

```

> 🛑 **Operational Posture Warning:** Once a version is destroyed, its state enum transitions to `DESTROYED`. The tracking slot remains in the ledger history to maintain audit trail continuity, but any future attempts to call `access_secret_version` against that specific path will fail with a gRPC error.

---

## Summary

In this lesson, we mastered the advanced data management and administrative capabilities of Google Cloud Secret Manager:

* Implemented systemic inventories across project scopes using `list_secrets`.
* Leveraged `FieldMask` configurations to modify key-value metadata taxonomies cleanly without causing property side effects.
* Extracted and audited historical version matrices using the explicit index pattern.
* Analyzed the risk profiles and execution differences behind container-level deletion and version-level data payload destruction.

Now, let's step over to the console code terminal and put these advanced cloud engineering patterns to work inside our interactive lab workspace!

## Google Cloud Secret Manager Operations with Python

Welcome to this task! You will be running a pre-existing Python script that exercises various operations with Google Cloud Secret Manager. This script demonstrates creating versions of a secret, updating the secret with both manually specified and randomly generated passwords, listing the secret versions, retrieving a secret's value by different version numbers, managing secret labels, and permanently deleting a secret. The purpose of this task is to expose you to how these operations work within a real script. There's no need for you to write any code—your task is simply to run the script, observe its execution and understand what each part of the code is doing.

Important Note: Executing scripts can alter resources within our GCP simulator. To return to the initial state, you may use the reset button located in the top right corner. It is important to note, however, that resetting will remove any code modifications. To safeguard your code during a reset, consider copying it to the clipboard.

```python
import mock_secret_manager
from google.cloud import secretmanager
import secrets
import string

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()
project_id = "your-project-id"  # Replace with your actual project ID
secret_id = "MyPassword"
parent = f"projects/{project_id}"
secret_name = f"{parent}/secrets/{secret_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": secret_id,
        "secret": {"replication": {"automatic": {}}},
    }
)

# Create version 1
version1 = client.add_secret_version(
    request={
        "parent": secret_name,
        "payload": {"data": b"simple password 1"},
    }
)

# Create version 2
version2 = client.add_secret_version(
    request={
        "parent": secret_name,
        "payload": {"data": b"simple password 2"},
    }
)

# Generate a strong random password for version 3
def generate_strong_password(length=16):
    alphabet = string.ascii_letters + string.digits + "!@#$%^&*"
    return ''.join(secrets.choice(alphabet) for _ in range(length))

strong_password = generate_strong_password(16)

# Create version 3 with the generated password
version3 = client.add_secret_version(
    request={
        "parent": secret_name,
        "payload": {"data": strong_password.encode()},
    }
)

# List Secret versions
versions = client.list_secret_versions(request={"parent": secret_name})
print('Versions of Secret:')
for version in versions:
    print(f"  {version.name}")

# Retrieve the previous version (version 2) explicitly
version2_name = f"{secret_name}/versions/2"
previous_secret_value = client.access_secret_version(request={"name": version2_name})
print(f'\nPrevious Secret Value (version 2): {previous_secret_value.payload.data.decode()}')

# Add labels to the secret (equivalent to AWS tags)
secret_with_labels = client.update_secret(
    request={
        "secret": {
            "name": secret_name,
            "labels": {"environment": "test"}
        },
        "update_mask": {"paths": ["labels"]}
    }
)
print(f'\nLabeled Secret: {secret_with_labels.name}')

# Remove labels from the secret
secret_without_labels = client.update_secret(
    request={
        "secret": {
            "name": secret_name,
            "labels": {}
        },
        "update_mask": {"paths": ["labels"]}
    }
)
print(f'\nUnlabeled Secret: {secret_without_labels.name}')

# Delete the secret permanently (no restoration possible in Google Cloud Secret Manager)
client.delete_secret(request={"name": secret_name})
print(f'\nDeleted Secret: {secret_name}')

```

## Secure Password Generation for Google Cloud Secret Manager

## Update Google Cloud Secret Manager with Auto-Generated Secure Password

## Labeling Secrets with Creator Information in Google Cloud Secret Manager

## Retrieving and Accessing Secret Versions

## Restoring Labels on a Secret

## Restoring Labels on a Secret